# Geometria OCR: zamrozona proba 12 nowych kolekcji
Wybierz GPU i kliknij **Uruchom wszystko**. Niczego nie wgrywaj recznie. Ostatnia komorka pobierze ZIP wynikow.
Proba zostala przypieta na GitHubie przed OCR: 12 regionow, 12 nowych kolekcji, 67 wierszy referencyjnych. Obrazy sa pobierane z przypietej rewizji Hugging Face i sprawdzane SHA256. Oba warianty same wykrywaja linie; liczba linii referencyjnych nie steruje segmentacja i zaden region nie wypada z mianownika.
Pisownia historyczna pozostaje bez zmian. Referencje sa upstream i nie przeszly recznej weryfikacji. To test deweloperski geometrii, nie benchmark modelu ani dowod SOTA.


In [ ]:
%pip install -q --upgrade transformers==4.57.6 jiwer==4.0.0 huggingface_hub==0.36.0 sentencepiece==0.2.1 opencv-python-headless==4.12.0.88

In [ ]:
"""Experimental image-only component grouping for single-column paragraph crops."""
from statistics import median


def detect_lines(image, *, follow_lines=False):
    """Return proposed boxes without accepting reference text, counts or line IDs."""
    import cv2
    import numpy as np

    gray = np.asarray(image.convert('L'))
    height, width = gray.shape
    binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    _, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    components = [tuple(map(int, s)) for s in stats[1:]]
    candidates = [h for x, y, w, h, area in components
                  if 8 <= h <= height * .45 and 2 <= w <= width * .08 and area >= 16]
    if not candidates:
        return {'boxes': [], 'foreign_ink_fraction': [], 'character_height': None, 'rejected_edge_groups': 0}
    ch = float(np.percentile(candidates, 65))
    anchors = [c for c in components if .5 * ch <= c[3] <= 1.8 * ch
               and c[4] >= .035 * ch * ch and c[2] <= width * .35]
    groups = []
    for c in sorted(anchors, key=lambda c: (c[0], c[1])):
        cy = c[1] + c[3] / 2
        cx = c[0] + c[2] / 2
        choices = []
        for i, group in enumerate(groups):
            xs = [p[0] + p[2] / 2 for p in group['parts']]
            if len(xs) >= 4 and max(xs) - min(xs) >= 3 * ch:
                slope, _ = np.polyfit(xs, group['centers'], 1)
                predicted = float(np.clip(slope, -.2, .2)) * (cx - median(xs)) + median(group['centers'])
            else:
                predicted = median(group['centers'])
            choices.append((abs(cy - predicted), i))
        if choices and min(choices)[0] <= .65 * ch:
            group = groups[min(choices)[1]]
            group['parts'].append(c)
            group['centers'].append(cy)
        else:
            groups.append({'parts': [c], 'centers': [cy]})

    def bounds(parts):
        return [min(c[0] for c in parts), min(c[1] for c in parts),
                max(c[0] + c[2] for c in parts), max(c[1] + c[3] for c in parts)]

    kept, rejected = [], 0
    for group in groups:
        parts = group['parts']
        box = bounds(parts)
        if len(parts) < 2 or box[2] - box[0] < 1.5 * ch:
            continue
        touches = sum(c[1] <= 1 or c[1] + c[3] >= height - 1 for c in parts)
        if touches / len(parts) >= .5:
            rejected += 1
            continue
        kept.append({'parts': parts[:], 'anchor_box': box})
    for group in kept:
        xs = [p[0] + p[2] / 2 for p in group['parts']]
        ys = [p[1] + p[3] / 2 for p in group['parts']]
        slope = float(np.clip(np.polyfit(xs, ys, 1)[0], -.2, .2)) if len(set(xs)) > 1 else 0.0
        group['fit'] = (slope, float(median([y - slope * x for x, y in zip(xs, ys)])))
    anchor_set = set(anchors)
    for c in components:
        x, y, w, h, area = c
        if c in anchor_set or area < max(2, .0015 * ch * ch) or h > .65 * ch:
            continue
        if y <= 0 or y + h >= height:
            continue
        choices = []
        for i, group in enumerate(kept):
            a, b, d, e = group['anchor_box']
            margin = .8 * ch if follow_lines else .25 * ch
            if x + w < a - margin or x > d + margin:
                continue
            if follow_lines:
                slope, intercept = group['fit']
                center = slope * (x + w / 2) + intercept
                b, e = center - ch / 2, center + ch / 2
            gap = max(b - (y + h), y - e, 0)
            limit = .2 * ch if y >= e else .55 * ch
            if gap <= limit:
                distance = abs(y + h / 2 - (b + e) / 2)
                choices.append((gap, distance, i))
        if choices:
            kept[min(choices)[2]]['parts'].append(c)
    padding = max(2, round(.1 * ch))
    proposals = []
    component_labels = {c: i + 1 for i, c in enumerate(components)}
    for group in kept:
        x1, y1, x2, y2 = bounds(group['parts'])
        box = [max(0, x1 - padding), max(0, y1 - padding),
               min(width, x2 + padding), min(height, y2 + padding)]
        a, b, d, e = box
        roi = labels[b:e, a:d]
        ink = int(np.count_nonzero(roi))
        own = int(np.count_nonzero(np.isin(roi, [component_labels[c] for c in group['parts']])))
        proposals.append((box, (ink - own) / ink if ink else 1.0, group))
    proposals.sort(key=lambda p: ((p[0][1] + p[0][3]) / 2, p[0][0]))
    result = {'boxes': [p[0] for p in proposals], 'foreign_ink_fraction': [p[1] for p in proposals],
              'character_height': ch, 'rejected_edge_groups': rejected}
    if follow_lines:
        bands = []
        for index, (box, _, group) in enumerate(proposals):
            a, b, d, e = box
            xs = np.arange(a, d)
            slope, intercept = group['fit']
            centers = slope * xs + intercept
            top = np.full(d - a, b, dtype=int)
            bottom = np.full(d - a, e, dtype=int)
            if index:
                s, t = proposals[index - 1][2]['fit']
                top = np.maximum(top, np.ceil((centers + s * xs + t) / 2).astype(int))
            if index + 1 < len(proposals):
                s, t = proposals[index + 1][2]['fit']
                bottom = np.minimum(bottom, np.floor((centers + s * xs + t) / 2).astype(int))
            # Never trim an assigned component, including detached accents.
            for x, y, w, h, _ in group['parts']:
                left, right = max(a, x - padding) - a, min(d, x + w + padding) - a
                top[left:right] = np.minimum(top[left:right], max(b, y - padding))
                bottom[left:right] = np.maximum(bottom[left:right], min(e, y + h + padding))
            bands.append({'top': top.clip(b, e).tolist(), 'bottom': bottom.clip(b, e).tolist()})
        result['line_bands'] = bands
    return result


def crop_line_band(image, box, band):
    """Whiten outside a proposed band; preserve source pixels inside it."""
    import numpy as np
    from PIL import Image
    a, b, d, e = box
    top, bottom = np.asarray(band['top']), np.asarray(band['bottom'])
    if top.shape != (d - a,) or bottom.shape != (d - a,) or np.any(top > bottom):
        raise ValueError('Invalid line band')
    pixels = np.array(image.convert('RGB').crop(box))
    ys = np.arange(b, e)[:, None]
    pixels[(ys < top) | (ys >= bottom)] = 255
    return Image.fromarray(pixels)

"""Reject missing, drifted or stale notebook dependencies before inference."""
import importlib.metadata
import sys

REQUIRED_PACKAGES = {
    'transformers': '4.57.6', 'huggingface_hub': '0.36.0',
    'jiwer': '4.0.0', 'sentencepiece': '0.2.1',
}


def check_colab_environment(extra=None):
    required = {**REQUIRED_PACKAGES, **(extra or {})}
    problems = []
    for name, expected in required.items():
        try:
            installed = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            installed = 'missing'
        if installed != expected:
            problems.append(f'{name}: installed={installed}, required={expected}')
    for name in [*required, 'tokenizers']:
        loaded = sys.modules.get(name)
        version = getattr(loaded, '__version__', None)
        if version is not None:
            try:
                installed = importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                installed = 'missing'
            if version != installed:
                problems.append(f'{name}: loaded={version}, installed={installed}; restart required')
    if problems:
        raise RuntimeError('Environment check failed. Run the install cell, restart the Colab session, '
                           'then Run all.\n' + '\n'.join(problems))
    print('Pinned dependencies verified; no stale package versions detected.')

"""Collection-disjoint region OCR diagnostic for two image-only geometries."""
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import base64
import gc
import hashlib
import importlib.metadata
import io
import json
import unicodedata
from urllib.request import urlopen
import zipfile

DATASET = 'PiotrSty/impact-psnc-polish-ocr'
REVISION = 'c7cb156fb95d2880699c33725bbaf1fbc1008fea'
MODELS = {'microsoft/trocr-base-printed': '93450be3f1ed40a930690d951ef3932687cc1892', 'PiotrSty/trocr-pl-mixed-v3': '85d0c91c26f8e088849096dded7c9ba10b4cd9c9'}
FROZEN = {'NA2_FT', 'Nowiny_z_Rakuz_FT', 'Powodzenia_FT'}

VARIANTS = ('rectangle', 'line_band')


def digest(data):
    return hashlib.sha256(data).hexdigest()


def normalize(text):
    return ' '.join(unicodedata.normalize('NFC', text).split())


def load_manifest(data, expected_sha256):
    if digest(data) != expected_sha256:
        raise ValueError('Manifest checksum mismatch')
    manifest = json.loads(data)
    if manifest['scope'] != 'geometry holdout diagnostic; not benchmark or model holdout':
        raise ValueError('Unexpected scope')
    if manifest['dataset'] != DATASET or manifest['revision'] != REVISION:
        raise ValueError('Dataset provenance mismatch')
    rows = manifest['regions']
    if len(rows) != 12 or len({r['id'] for r in rows}) != len(rows):
        raise ValueError('Expected 12 unique regions')
    if len({r['collection'] for r in rows}) != len(rows):
        raise ValueError('Expected one region per collection')
    for row in rows:
        path = PurePosixPath(row['source_path'])
        if path.is_absolute() or '..' in path.parts or '\\' in row['source_path'] or ':' in row['source_path']:
            raise ValueError('Unsafe source path')
        if row['split'] != 'train' or row['collection'] in FROZEN:
            raise ValueError('Invalid holdout split')
        if row['eligible_for_benchmark'] is not False or '\ufffd' in row['text']:
            raise ValueError('Invalid reference status')
    return manifest


def fetch_sources(rows, opener=urlopen):
    images = []
    root = f'https://huggingface.co/datasets/{DATASET}/resolve/{REVISION}/'
    for row in rows:
        with opener(root + row['source_path'], timeout=120) as response:
            content = response.read()
        if digest(content) != row['image_sha256']:
            raise ValueError('Source checksum mismatch: ' + row['id'])
        images.append(content)
    return images


def segment(content, variant):
    from PIL import Image
    if variant not in VARIANTS:
        raise ValueError('Unknown geometry variant')
    with Image.open(io.BytesIO(content)) as source:
        source = source.convert('RGB')
        result = detect_lines(source, follow_lines=variant == 'line_band')
        crops = []
        for index, box in enumerate(result['boxes']):
            crop = (crop_line_band(source, box, result['line_bands'][index])
                    if variant == 'line_band' else source.crop(box))
            crops.append(crop)
    return result, crops


def score(rows, predictions):
    from jiwer import cer, wer, process_characters
    if [r['id'] for r in rows] != [p['id'] for p in predictions]:
        raise ValueError('Prediction/reference IDs differ')
    result = {}
    subsets = {'all_regions': list(range(len(rows))),
               'without_private_use_references': [i for i, r in enumerate(rows)
                                                   if r['reference_private_use_count'] == 0]}
    for label, indices in subsets.items():
        refs = [normalize(rows[i]['text']) for i in indices]
        hyps = [normalize(predictions[i]['text']) for i in indices]
        result[label] = {'regions': len(indices), 'reference_characters': sum(len(x) for x in refs),
                         'cer': cer(refs, hyps), 'wer': wer(refs, hyps),
                         'lowercase_cer_diagnostic': cer([x.lower() for x in refs], [x.lower() for x in hyps]),
                         'errors': sum(predictions[i]['status'] != 'ok' for i in indices),
                         'empty': sum(not x for x in hyps),
                         'detected_lines': sum(predictions[i]['detected_lines'] for i in indices)}
    edits = []
    for row, pred in zip(rows, predictions):
        value = process_characters(normalize(row['text']), normalize(pred['text']))
        edits.append(value.substitutions + value.deletions + value.insertions)
    result['per_region_character_edits'] = dict(zip([r['id'] for r in rows], edits))
    return result


def compare_variants(rows, rectangle, line_band):
    a = score(rows, rectangle)['per_region_character_edits']
    b = score(rows, line_band)['per_region_character_edits']
    deltas = {row['id']: b[row['id']] - a[row['id']] for row in rows}
    return {'improved': sum(x < 0 for x in deltas.values()),
            'regressed': sum(x > 0 for x in deltas.values()),
            'tied': sum(x == 0 for x in deltas.values()),
            'net_character_edit_change': sum(deltas.values()), 'deltas': deltas}


def run(manifest_data, expected_sha256, runner_sha256=None):
    manifest = load_manifest(manifest_data, expected_sha256)
    rows = manifest['regions']
    image_bytes = fetch_sources(rows)
    import torch
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    if not torch.cuda.is_available():
        raise RuntimeError('Select GPU runtime in Colab, then Run all')
    torch.manual_seed(0)
    output = Path('/content') / ('geometry-holdout-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    output.mkdir(parents=True)
    def save(name, value):
        (output / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    save('input-manifest.json', manifest)
    provenance = {'source_revision': REVISION, 'manifest_sha256': expected_sha256,
                  'runner_sha256': runner_sha256, 'reference_text_sent_to_model': False,
                  'selection_frozen_before_ocr': True, 'all_regions_retained': True,
                  'segmentation_uses_reference_line_count': False,
                  'scope': manifest['scope'], 'limitations': manifest['limitations']}
    save('provenance.json', provenance)
    report = {'scope': manifest['scope'], 'models': MODELS, 'gpu': torch.cuda.get_device_name(0),
              'environment': {p: importlib.metadata.version(p) for p in
                              ['torch', 'transformers', 'Pillow', 'jiwer', 'huggingface_hub',
                               'opencv-python-headless']},
              'generation': {'do_sample': False, 'num_beams': 1, 'max_new_tokens': 256, 'dtype': 'float32'},
              'normalization': manifest['normalization'], 'results': {}}
    for model_id, revision in MODELS.items():
        processor = model = None
        by_variant = {variant: [] for variant in VARIANTS}
        try:
            processor = TrOCRProcessor.from_pretrained(model_id, revision=revision, trust_remote_code=False)
            model = VisionEncoderDecoderModel.from_pretrained(
                model_id, revision=revision, trust_remote_code=False).float().cuda().eval()
            eos = model.generation_config.eos_token_id
            eos = set(eos if isinstance(eos, list) else [eos])
            for variant in VARIANTS:
                for row, content in zip(rows, image_bytes):
                    prediction = {'id': row['id'], 'variant': variant, 'text': '', 'status': 'ok',
                                  'detected_lines': 0, 'lines': []}
                    try:
                        geometry, crops = segment(content, variant)
                        prediction['detected_lines'] = len(crops)
                        prediction['geometry'] = {'boxes': geometry['boxes'],
                                                  'foreign_ink_fraction': geometry['foreign_ink_fraction']}
                        if not crops:
                            raise ValueError('No lines detected')
                        texts = []
                        for index, crop in enumerate(crops):
                            pixels = processor(images=crop, return_tensors='pt').pixel_values.cuda()
                            with torch.inference_mode():
                                ids = model.generate(pixels, do_sample=False, num_beams=1,
                                                     max_new_tokens=256)[0].tolist()
                            text = processor.batch_decode([ids], skip_special_tokens=True)[0]
                            texts.append(text)
                            prediction['lines'].append({'index': index, 'text': text, 'token_ids': ids,
                                                        'ended_with_eos': ids[-1] in eos,
                                                        'possibly_truncated': len(ids) >= 257 and ids[-1] not in eos})
                        prediction['text'] = '\n'.join(texts)
                    except Exception as exc:
                        prediction['status'] = 'error'
                        prediction['error'] = type(exc).__name__ + ': ' + str(exc)
                    by_variant[variant].append(prediction)
        except Exception as exc:
            for variant in VARIANTS:
                by_variant[variant] = [{'id': r['id'], 'variant': variant, 'text': '', 'status': 'error',
                                        'detected_lines': 0, 'lines': [],
                                        'error': type(exc).__name__ + ': ' + str(exc)} for r in rows]
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()
        save(model_id.replace('/', '--') + '.json', by_variant)
        report['results'][model_id] = {variant: score(rows, by_variant[variant]) for variant in VARIANTS}
        report['results'][model_id]['comparison'] = compare_variants(
            rows, by_variant['rectangle'], by_variant['line_band'])
        save('report.json', report)
        print(model_id, json.dumps(report['results'][model_id], indent=2))
    save('checksums.json', {p.name: digest(p.read_bytes()) for p in output.iterdir() if p.is_file()})
    archive = output.with_suffix('.zip')
    with zipfile.ZipFile(archive, 'x', zipfile.ZIP_DEFLATED) as bundle:
        for path in output.iterdir():
            bundle.write(path, path.name)
    from IPython.display import HTML, display
    encoded = base64.b64encode(archive.read_bytes()).decode('ascii')
    display(HTML('<a download="' + archive.name + '" href="data:application/zip;base64,' + encoded +
                 '">Pobierz ZIP wynikow</a>'))
    print('Output:', archive)
    return archive

check_colab_environment({'opencv-python-headless': '4.12.0.88'})
MANIFEST_URL = ('https://raw.githubusercontent.com/PiotrStyla/OCR_engine/'
                '63c922624e8aeed10113a1060615f1dc74b962eb/experiments/2026-09-24/geometry-holdout-v1/manifest.json')
with urlopen(MANIFEST_URL, timeout=120) as response:
    manifest_data = response.read()
result_archive = run(manifest_data, '0b5459ea56d352de532b2cdc2284bd701b1c98458c67afd9f2220b6f0a6a7b18', 'd7f5d6c1bd48680a27729a86cb3029810ec2b84210061206968d286fc9712180')
with zipfile.ZipFile(result_archive) as evidence:
    final_report = json.loads(evidence.read('report.json'))
failed = []
for model, result in final_report['results'].items():
    for variant in VARIANTS:
        score_result = result[variant]['all_regions']
        if score_result['errors']:
            failed.append(f"{model} / {variant}: {score_result['errors']} region errors")
        else:
            print(f"{model} / {variant}: CER={score_result['cer']:.2%}, "
                  f"WER={score_result['wer']:.2%}, lines={score_result['detected_lines']}")
    print(model, result['comparison'])
if failed:
    print('INCOMPLETE RUN. Error-derived CER is not model quality. Download the diagnostic ZIP.\n' +
          '\n'.join(failed))
else:
    print('COMPLETE: 12/12 regions retained in both variants. Development holdout, not SOTA evidence.')


In [ ]:
from google.colab import files
files.download(str(result_archive))
